# Classification NBA Model

## Configuration

## Imports

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [ ]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [6]:
import time
time.sleep(4*3600)

## Load Data

In [7]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260312.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [8]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50, max_na_per_row=5, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11202 rows
Basic cleaning complete: 8593 rows remaining

Starting advanced column cleaning with 3243 columns

Advanced column cleaning complete: 3243 → 2134 columns (1109 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 5 NaN values...
Removed 3945 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4648, 2134)


In [9]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1943,total_consensus_pct_under,67,1.44,2024.0
1944,spread_consensus_pct_home,63,1.36,2024.0
1950,ml_consensus_opener_price_away,14,0.30,2025.0
1951,ml_consensus_opener_price_home,14,0.30,2025.0
1803,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1799,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,9,0.19,2023.0
1805,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1801,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,8,0.17,2023.0
1221,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,5,0.11,2023.0
555,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,5,0.11,2023.0


In [10]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [11]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [12]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1158
2022    1173
2023     170
2024    1248
2025     899
dtype: int64


## Train / Test

In [13]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [14]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4411
Final test set size: 237
Final test date range: 2026-02-03 00:00:00 -> 2026-03-11 00:00:00


In [15]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4411, 2131)
X_test_final shape: (237, 2131)


In [16]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [17]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=100,
    train_games=3500,
    min_train_games=2000,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2776            35       2021-10-29     2024-12-07      2024-12-08    2024-12-15         2024
    2           2913            32       2021-10-29     2024-12-31      2025-01-01    2025-01-04         2024
    3           3046            37       2021-10-29     2025-01-18      2025-01-19    2025-01-23         2024
    4           3188            31       2021-10-29     2025-02-06      2025-02-07    2025-02-10         2024
    5           3322            33       2021-10-29     2025-03-01      2025-03-02    2025-03-05         2024
    6           3455            31       2021-10-29     2025-03-18      2025-03-19    2025-03-22         2024
    7           3500            38       2021-11-13     2025-04-04      2025-04-05    2025-04-09         2024
    8           3500            32       2021-12-06     2025-06-22      2025

In [18]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 381.17048
Validation MSE: 358.39178

Train RMSE: 19.52331
Validation RMSE: 18.87951

Train MAE: 15.58023
Validation MAE: 14.84995

Train R2: 0.00000
Validation R2: -0.09993

Train OU_Betting_Accuracy: 49.56%
Validation OU_Betting_Accuracy: 48.35%



In [19]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 160.63416
Validation MSE: 46539901.69131

Train RMSE: 12.66571
Validation RMSE: 2008.81302

Train MAE: 9.84492
Validation MAE: 1906.61071

Train R2: 0.57830
Validation R2: -169924.59130

Train OU_Betting_Accuracy: 73.73%
Validation OU_Betting_Accuracy: 45.77%



In [20]:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=750,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 7.25078
Validation MSE: 296.33744

Train RMSE: 2.65950
Validation RMSE: 17.12733

Train MAE: 2.04288
Validation MAE: 13.82783

Train R2: 0.98093
Validation R2: 0.08016

Train OU_Betting_Accuracy: 97.59%
Validation OU_Betting_Accuracy: 49.34%



In [21]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 331.35599
RMSE: 18.20319
MAE: 14.35276
OU_Betting_Accuracy: 50.00%


In [22]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,237,100.0%,50.00%
1,1,208,87.8%,50.00%
2,2,168,70.9%,51.83%
3,3,128,54.0%,56.80%
4,4,98,41.4%,54.74%
5,5,70,29.5%,55.07%
6,6,53,22.4%,55.77%
7,7,41,17.3%,55.00%
8,8,31,13.1%,50.00%
9,9,17,7.2%,41.18%


# OPTUNA

In [23]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=6 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-15 06:59:03,606] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-15 07:05:16,179] Trial 0 finished with value: 12.967550752279516 and parameters: {'max_depth': 2, 'min_child_weight': 14.839989016734002, 'gamma': 1.6521043697435434, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.6123279759064977, 'learning_rate': 0.015902381379451283, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 0.2766344129879233}. Best is trial 0 with value: 12.967550752279516.
[I 2026-03-15 07:11:50,415] Trial 1 finished with value: 12.967625528620436 and parameters: {'max_depth': 2, 'min_child_weight': 35.38241645406617, 'gamma': 1.6910441407044758, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.7751882300061779, 'learning_rate': 0.013902617405829187, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 0.6196026989845951}. Best is trial 0 with value: 12.967550752279516.
[I 2026-03-15 07:16:29,827] Trial 2 finished with value: 12.95707442064247 and parameters: {'max_depth': 4, 'min_child_weight': 13.12932060704323, 'gamma': 0.6451864311430185, 'subsample':

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,73,12.6685,16.1015,0.1925,54.57%,141,2,7.190105,2.358533,0.911552,0.731477,0.075118,0.084527,8.525672
1,37,12.6742,16.0215,0.1999,53.09%,155,2,5.715012,1.939228,0.930386,0.756904,0.065929,0.011060,2.031810
2,47,12.7224,16.1756,0.1853,55.24%,121,2,6.021227,1.623305,0.884788,0.692279,0.070345,0.020688,4.070987
3,51,12.7361,16.1420,0.1886,54.95%,131,2,7.774397,1.676206,0.871926,0.714548,0.079806,0.013375,3.319076
4,19,12.7490,16.2132,0.1811,53.75%,139,3,17.938939,2.564678,0.897388,0.650717,0.062082,0.566620,0.188219
5,66,12.7721,16.1701,0.1861,53.71%,132,2,9.221991,2.366841,0.924028,0.749278,0.069009,0.032688,7.568975
6,70,12.7725,16.2434,0.1788,54.41%,86,5,5.835986,2.222593,0.948747,0.705252,0.050365,0.020876,5.583901
7,46,12.7762,16.2994,0.1738,54.44%,90,2,5.980749,1.658352,0.863598,0.696533,0.072331,0.019599,4.963770
8,43,12.7792,16.1827,0.1837,53.32%,119,2,8.230739,1.745215,0.878193,0.725452,0.070370,0.088646,2.901390
9,44,12.7848,16.2547,0.1781,54.61%,81,2,8.620943,1.707241,0.867399,0.723371,0.077968,0.010124,3.269242


In [24]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 315.11539
RMSE: 17.75149
MAE: 14.01226
OU_Betting_Accuracy: 53.02%


In [ ]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

model_name = "recent_xgb_total_points_12_03_26"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version="12_03_26",
        model_type="recent_games_total_points",
        prediction_source="recent_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        train_date_min=df_dev["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_dev["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=best_model,
    feature_names=list(X_dev.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/",
    metadata=metadata,
)

print("Saved model :", model_path)
print("Saved metadata:", meta_path)
